In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 92.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=fedd014ec7633836aa875449035b6454c3232b0f117e20fa8ea704e6b39b4da6
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [2]:
def prepare_state(bit, basis):
    qc = QuantumCircuit(1, 1)
    if basis == 'Z': # Computational Basis
        if bit == 1:
            qc.x(0) # Apply X-gate to get |1>
    elif basis == 'X': # Hadamard Basis
        if bit == 0:
            qc.h(0) # Apply H-gate to get |+>
        else:
            qc.x(0) # Start with |1>
            qc.h(0) # Apply H-gate to get |->
    else:
        raise ValueError("Basis must be 'Z' or 'X'")
    return qc


def measure_state(qc, basis):
    if basis == 'X': # If measuring in X-basis, rotate to Z-basis first
        qc.h(0)
    qc.measure(0, 0)
    return qc


In [4]:
# Example: Prepare |0> in Z basis, measure in Z basis
qc_0_Z = prepare_state(0, 'Z')
measured_qc_0_Z = measure_state(qc_0_Z.copy(), 'Z')

# Example: Prepare |1> in X basis, measure in X basis
qc_1_X = prepare_state(1, 'X')
measured_qc_1_X = measure_state(qc_1_X.copy(), 'X')

# Example: Prepare |0> in Z basis, measure in X basis (to show error)
qc_0_Z_mis = prepare_state(0, 'Z')
measured_qc_0_Z_mis = measure_state(qc_0_Z_mis.copy(), 'X')

# Simulate and get results
simulator = BasicSimulator()

job_0_Z = simulator.run(transpile(measured_qc_0_Z, simulator), shots=1000)
counts_0_Z = job_0_Z.result().get_counts(measured_qc_0_Z)
print(f"Prepared |0> in Z, Measured in Z: {counts_0_Z}")

job_1_X = simulator.run(transpile(measured_qc_1_X, simulator), shots=1000)
counts_1_X = job_1_X.result().get_counts(measured_qc_1_X)
print(f"Prepared |1> in X, Measured in X: {counts_1_X}")

job_0_Z_mis = simulator.run(transpile(measured_qc_0_Z_mis, simulator), shots=1000)
counts_0_Z_mis = job_0_Z_mis.result().get_counts(measured_qc_0_Z_mis)
print(f"Prepared |0> in Z, Measured in X: {counts_0_Z_mis}")


Prepared |0> in Z, Measured in Z: {'0': 1000}
Prepared |1> in X, Measured in X: {'1': 1000}
Prepared |0> in Z, Measured in X: {'1': 505, '0': 495}
